# Team aEC/T Leaderboard

Rank UFA teams by possession aEC per throw. The main rate is `team_aec_per_throw = sum(total_aec) / sum(throw_count)`, which treats every throw as one unit in the team rate.

In [17]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ufa_aec_possessions
import ufa_aec_possessions.fetching
import ufa_aec_possessions.leaderboards

importlib.reload(ufa_aec_possessions.fetching)
importlib.reload(ufa_aec_possessions.leaderboards)
importlib.reload(ufa_aec_possessions)

from ufa_aec_possessions import (
    build_possessions,
    build_scoring_possessions,
    fetch_shownspace_season_throws_cached,
    filter_analysis_possessions,
    summarize_team_aec_per_throw,
)

In [18]:
SEASON = 2026
MAX_GAMES = None
FORCE_REFRESH = False
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'shownspace'
MIN_POSSESSIONS = 20
TOP_TEAMS = 22

## Load League Possessions

In [19]:
league_games, league_throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)
league_all_possessions, league_all_paths = build_possessions(league_throws)
league_possessions, league_paths = build_scoring_possessions(league_throws)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'League all possessions found: {len(league_all_possessions):,}')
print(f'League scoring possessions found: {len(league_possessions):,}')

League games loaded: 118
League throws loaded: 64,964
League all possessions found: 9,214
League scoring possessions found: 4,673


In [ ]:
LEADERBOARD_COLUMNS = [
    'rank', 'team_id', 'team_aec_per_throw', 'avg_possession_aec_per_throw',
    'possessions', 'games', 'throws', 'total_aec', 'avg_throw_count',
    'avg_field_progress', 'huck_rate', 'resets_per_possession', 'o_line_share',
]

FORMATTERS = {
    'team_aec_per_throw': '{:.3f}',
    'avg_possession_aec_per_throw': '{:.3f}',
    'total_aec': '{:.1f}',
    'avg_throw_count': '{:.1f}',
    'avg_field_progress': '{:.1f}',
    'huck_rate': '{:.1%}',
    'resets_per_possession': '{:.1f}',
    'o_line_share': '{:.1%}',
}


def show_team_aec_t_table(leaderboard, n=TOP_TEAMS):
    table = leaderboard.head(n).reindex(columns=LEADERBOARD_COLUMNS).copy()
    for column, formatter in FORMATTERS.items():
        if column in table:
            values = pd.to_numeric(table[column], errors='coerce')
            table[column] = values.map(lambda value: '' if pd.isna(value) else formatter.format(value))
    return table


def plot_team_aec_t_leaderboard(leaderboard, title, n=TOP_TEAMS):
    chart = leaderboard.head(n).sort_values('team_aec_per_throw', ascending=True).copy()
    chart['team_label'] = chart['rank'].astype(int).astype(str) + '. ' + chart['team_id'].astype(str).str.title()
    fig = px.bar(
        chart,
        x='team_aec_per_throw',
        y='team_label',
        orientation='h',
        text='team_aec_per_throw',
        hover_data={
            'team_label': False,
            'team_aec_per_throw': ':.3f',
            'avg_possession_aec_per_throw': ':.3f',
            'possessions': ':,',
            'games': ':,',
            'throws': ':,',
            'huck_rate': ':.1%',
            'o_line_share': ':.1%',
        },
        labels={'team_aec_per_throw': 'team aEC/T', 'team_label': 'Team'},
        color='team_aec_per_throw',
        color_continuous_scale='Greens',
    )
    fig.update_traces(texttemplate='%{x:.3f}', textposition='outside', cliponaxis=False)
    fig.update_layout(
        title={'text': title, 'x': 0.5, 'xanchor': 'center'},
        height=max(520, 30 * len(chart) + 160),
        width=900,
        margin={'l': 110, 'r': 40, 't': 70, 'b': 50},
        coloraxis_showscale=False,
        plot_bgcolor='#f6faf5',
        paper_bgcolor='#ffffff',
    )
    return fig

## All Possessions Including Turnovers

This is the closest view to Shown Space team `Tot-aEC`: it includes scoring possessions, turnovers, and other built possessions. This is the view where Bighorns show up as negative overall.

In [ ]:
team_aec_t_all_possessions = summarize_team_aec_per_throw(
    league_all_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_possessions,
    title=f'{SEASON} UFA team aEC/T - all possessions including turnovers',
)

In [ ]:
show_team_aec_t_table(team_aec_t_all_possessions)

## Total aEC Sanity Check

Shown Space's `Tot-aEC` is a total, not an aEC/T rate. Sorting the all-possession summary by `total_aec` should line up with that table much more closely than the scoring-only views.

In [ ]:
team_total_aec_low_to_high = team_aec_t_all_possessions.sort_values('total_aec', ascending=True).reset_index(drop=True).copy()
team_total_aec_low_to_high['rank'] = range(1, len(team_total_aec_low_to_high) + 1)
show_team_aec_t_table(team_total_aec_low_to_high)

## All Scoring Possessions

This view only includes possessions that ended in goals. It answers a narrower question: when a team scores, how much aEC does that scoring possession create per throw? It should not be read as overall team aEC.

In [ ]:
team_aec_t_all_scoring = summarize_team_aec_per_throw(
    league_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_scoring,
    title=f'{SEASON} UFA team aEC/T - all scoring possessions',
)

In [ ]:
show_team_aec_t_table(team_aec_t_all_scoring)

## Default Possession-Shape Pool

This uses the same default filter as the top-possession shape work: O-line scoring possessions, long-field starts, hucks excluded.

In [ ]:
analysis_possessions, analysis_paths = filter_analysis_possessions(
    league_possessions,
    league_paths,
)
team_aec_t_default_pool = summarize_team_aec_per_throw(
    analysis_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_default_pool,
    title=f'{SEASON} UFA team aEC/T - O-line non-huck long-field scores',
)

In [ ]:
show_team_aec_t_table(team_aec_t_default_pool)